# Data Visualisation Fundamentals: Choosing, Building and Interpreting Charts

**Data Science & Machine Learning: from Skills to the Project** · Session 1 · 90 minutes

This notebook is a practical introduction to choosing and building charts. Keep returning to one question: *what does this chart let me see, and what might it hide?*

The dataset is **synthetic**. It is generated inside this notebook with `numpy.random.default_rng(42)`, so it is reproducible and contains no real customers, products or sales. Run the cells from top to bottom.

## Learning outcomes

By the end of the session you should be able to:

- choose a chart type from the variable type and the analytical question;
- build static plots with Matplotlib's object-oriented API and Seaborn;
- distinguish exploratory charts (for you) from explanatory charts (for an audience);
- create and interpret categorical, distribution and relationship plots;
- identify misleading scales, inappropriate chart types and poor colour choices.

## Agenda (90 minutes)

| Time | Section | Focus |
|---|---|---|
| 5 min | 1. Introduction | Why visualise; the workflow; similar statistics, different shapes |
| 10 min | 2. Chart selection | Variable types, analytical questions, visual encoding |
| 10 min | 3. Matplotlib fundamentals | Figure, Axes, labels, units, layout |
| 10 min | 4. Categorical comparisons | Bars, ordering, zero baselines, pie charts |
| 15 min | 5. Numerical distributions | Histograms, KDE, ECDF, box and violin plots |
| 15 min | 6. Relationships and trends | Scatter plots, overplotting, line plots, hexbin |
| 10 min | 7. Visual integrity | Misleading scales and colours; accessible design |
| 10 min | 8. Guided practical | Three retail questions with starter code |
| 5 min | 9. Reflection | Five questions and key takeaways |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')

# Okabe-Ito palette: designed to stay distinguishable for most forms of colour blindness
OKABE_ITO = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7', '#56B4E9', '#F0E442', '#000000']
CATEGORY_COLOURS = dict(zip(['Electronics', 'Home', 'Clothing', 'Grocery', 'Sports'], OKABE_ITO[:5]))

## 1. Introduction: why visualise?

Summary statistics compress data, and compression can hide structure. A visualisation shows structure directly, but it can also mislead. Good analysis follows a loop:

**Question → Plot → Interpret → Check → Act**

- *Exploratory* charts are for you. They are quick, numerous and often rough. Their job is to surprise you.
- *Explanatory* charts are for an audience. They are fewer, polished and built around one message.

The demonstration below uses Anscombe's quartet (Anscombe, 1973): four small datasets with almost identical means, variances and correlations.

In [ ]:
x_main = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5])
x_outlier = np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8])
quartet = {
    'A': (x_main, np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])),
    'B': (x_main, np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])),
    'C': (x_main, np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])),
    'D': (x_outlier, np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])),
}

summary = pd.DataFrame({
    name: {'mean x': x.mean(), 'mean y': y.mean(), 'var y': y.var(ddof=1), 'correlation': np.corrcoef(x, y)[0, 1]}
    for name, (x, y) in quartet.items()
}).T.round(2)
print(summary)

fig, axes = plt.subplots(2, 2, figsize=(8, 6), sharex=True, sharey=True)
for ax, (name, (x, y)) in zip(axes.flat, quartet.items()):
    ax.scatter(x, y, color=OKABE_ITO[0])
    ax.set_title(f'Dataset {name}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
fig.suptitle('Same summary statistics, different structure', y=1.02)
fig.tight_layout()

**Observation (not a causal claim):** All four datasets share the same summary statistics, yet only dataset A looks like the straight-line relationship those statistics suggest. B is curved, C has one point that pulls the fitted line away from the rest, and D's apparent slope depends entirely on a single point.

**Question:** If you had seen only the table, what decision might you have made about each dataset? Which plot would have prevented the mistake?

## Setup: the synthetic retail dataset

The next cell creates a synthetic retail dataset of 360 sale records. It is generated in the notebook, so no files or internet access are needed, and every run produces identical values. It deliberately includes:

- **category differences**: Electronics are expensive and sell fewer units; Grocery is cheap and sells many (by design);
- **a seasonal pattern**: sales are higher in late autumn;
- **outliers**: three premium Electronics items and four bulk orders.

Revenue is defined as price × units sold.

In [ ]:
rng = np.random.default_rng(42)
n = 360

# Categorical variables
category = rng.choice(['Electronics', 'Home', 'Clothing', 'Grocery', 'Sports'], size=n,
                      p=[0.18, 0.24, 0.26, 0.22, 0.10])
region = rng.choice(['North', 'South', 'East', 'West'], size=n)

# Price: a category-specific typical level with multiplicative (log-normal) variation
typical_price = pd.Series(category).map({'Electronics': 180, 'Home': 55, 'Clothing': 32,
                                         'Grocery': 5, 'Sports': 45}).to_numpy()
price = typical_price * rng.lognormal(mean=0.0, sigma=0.25, size=n)
luxury = rng.choice(np.flatnonzero(category == 'Electronics'), size=3, replace=False)
price[luxury] *= 2.5                       # three premium items (outliers)
price = price.round(2)

# Sale date across 2025, with a stronger late-year season
day_offset = rng.integers(0, 365, size=n)
sale_date = pd.Timestamp('2025-01-01') + pd.to_timedelta(day_offset, unit='D')
season = 1 + 0.3 * np.cos(2 * np.pi * (day_offset - 330) / 365)   # peaks around late November

# Units sold: fall with price, rise with season, plus Poisson noise
expected_units = (60 * np.exp(-price / 90) + 8) * season
units_sold = rng.poisson(expected_units)
bulk_rows = rng.choice(n, size=4, replace=False)   # four bulk orders (outliers)
units_sold[bulk_rows] = rng.integers(180, 260, size=4)

# Customer rating: category-specific shift, bounded between 1 and 5
rating_shift = pd.Series(category).map({'Electronics': -0.2, 'Home': 0.1, 'Clothing': 0.0,
                                        'Grocery': 0.2, 'Sports': 0.0}).to_numpy()
customer_rating = np.clip(rng.normal(3.9, 0.5, size=n) + rating_shift, 1, 5).round(1)

df = pd.DataFrame({
    'product_id': [f'SKU-{i:04d}' for i in range(1, n + 1)],
    'category': category,
    'region': region,
    'price': price,
    'units_sold': units_sold,
    'customer_rating': customer_rating,
    'sale_date': sale_date,
})
df['revenue'] = (df['price'] * df['units_sold']).round(2)
df = df.sort_values('sale_date', ignore_index=True)

print(f'{df.shape[0]} records, {df.shape[1]} columns')
df.head()

## 2. Chart selection: variable types, questions and encoding

The right chart depends on two things: the **variable type** and the **analytical question**.

| Variable type | Examples in this dataset | Typical summaries |
|---|---|---|
| Nominal (categories, no order) | `category`, `region`, `product_id` | counts, proportions |
| Ordinal-like (ordered scale) | `customer_rating` (1–5) | medians, ordered counts |
| Numerical (continuous or count) | `price`, `units_sold`, `revenue` | distributions, means, spread |
| Temporal | `sale_date` | trends over time |

| Analytical question | Good first choice | Example question |
|---|---|---|
| Comparison across categories | sorted bar chart | Which category earns most per record? |
| Distribution of one variable | histogram, KDE, ECDF, box or violin | How spread out are the prices? |
| Relationship between two variables | scatter plot, hexbin | Do higher prices go with fewer units sold? |
| Change over time | line chart | Did revenue change through the year? |

**Visual encoding.** Cleveland and McGill's research (1984) suggests people judge positions on a common scale most accurately, then lengths, then angles and slopes. Areas, volumes and colour saturation are harder to compare. Prefer position and length where you can.

In [ ]:
variable_types = pd.DataFrame({
    'column': df.columns,
    'pandas dtype': df.dtypes.astype(str).to_numpy(),
    'analytical type': ['identifier (nominal)', 'nominal', 'nominal', 'numerical (ratio)',
                        'numerical (count)', 'ordinal-like rating (1–5)', 'temporal', 'numerical (ratio)'],
})
variable_types

## 3. Matplotlib fundamentals

Matplotlib has two interfaces. The pyplot style (`plt.plot(...)`) is convenient for quick work. The **object-oriented** style is clearer for reports and for anything with several panels:

- a **Figure** is the whole canvas;
- an **Axes** is one plotting area with its own axes, title and labels;
- you call methods on the Axes (`ax.plot`, `ax.set_title`, `ax.set_xlabel`) rather than global functions.

Always include units in axis labels, for example `Revenue (£ thousands)`. Make the title state the message, not only the topic.

In [ ]:
monthly_revenue = df.set_index('sale_date')['revenue'].resample('MS').sum()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(monthly_revenue.index, monthly_revenue / 1000, marker='o', color=OKABE_ITO[0], linewidth=2)
ax.set_title('Total revenue by month, 2025')
ax.set_xlabel('Month of sale')
ax.set_ylabel('Revenue (£ thousands)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

**Observation:** Monthly revenue moves around from month to month. It is higher in the second half of the year than in the first quarter. This chart shows the pattern, but it cannot tell you whether the pattern is seasonal, random or caused by specific orders.

**Question:** Which elements of this chart are essential, and which could you remove without losing information?

## 4. Categorical comparisons

For nominal categories:

- **Sort** bars by value, unless the categories have a natural order (for example, months or survey scale levels).
- **Start bar lengths at zero.** Bar length is the encoding, so a non-zero baseline exaggerates differences.
- **Do not connect categories with a line.** A line implies a trend between neighbouring categories that does not exist.
- **Pie charts** ask people to compare angles and areas, which is hard. They can work for two or three strongly different parts, but a bar chart usually answers the question better.

The left panel below shows the mean revenue per record for each category, sorted in descending order. The right panel shows the same means connected by a line, in alphabetical order, to illustrate the problem.

In [ ]:
category_revenue = df.groupby('category')['revenue'].mean().sort_values(ascending=False)
alphabetical = category_revenue.sort_index()

fig, (ax_bar, ax_line) = plt.subplots(1, 2, figsize=(11, 4))

bars = ax_bar.bar(category_revenue.index, category_revenue.values,
                  color=[CATEGORY_COLOURS[c] for c in category_revenue.index])
ax_bar.bar_label(bars, labels=[f'£{v:,.0f}' for v in category_revenue.values], padding=3)
ax_bar.set_ylim(0, category_revenue.max() * 1.15)   # bars start at zero
ax_bar.set_title('Better: sorted bars on a zero baseline')
ax_bar.set_ylabel('Mean revenue per record (£)')

ax_line.plot(alphabetical.index, alphabetical.values, marker='o', color='grey')
ax_line.set_ylim(0, category_revenue.max() * 1.15)
ax_line.set_title('Misleading: a line implies a trend between categories')
ax_line.set_ylabel('Mean revenue per record (£)')

**Observation:** Bar lengths are directly comparable because they share a zero baseline. The line in the right panel suggests a rise and fall between categories, but the alphabetical order is arbitrary, so the shape means nothing.

**Question:** Would the ranking of categories change if you used the median instead of the mean? Why might it?

The pie and bar comparison below shows the same regional split of units sold.

In [ ]:
region_units = df.groupby('region')['units_sold'].sum().sort_values(ascending=False)

fig, (ax_pie, ax_bar) = plt.subplots(1, 2, figsize=(10, 4))

ax_pie.pie(region_units.values, labels=region_units.index, autopct='%1.1f%%',
           colors=OKABE_ITO[:4], startangle=90, counterclock=False)
ax_pie.set_title('Pie: share of units sold')

ax_bar.barh(region_units.index[::-1], region_units.values[::-1], color=OKABE_ITO[0])
ax_bar.bar_label(ax_bar.containers[0], fmt='{:,.0f}', padding=3)
ax_bar.set_xlim(0, region_units.max() * 1.15)
ax_bar.set_xlabel('Units sold')
ax_bar.set_title('Bar: the same comparison on a common scale')

**Observation:** The pie slices look similar, and the percentages are hard to order by eye. The bar chart makes the ordering and the gaps easier to judge. These are descriptions of this sample; they do not explain why the regions differ.

**Question:** Which chart would you choose to show that one region sold about 5% more units than another? Why?

## 5. Numerical distributions

A distribution shows how values are spread. Each tool answers a slightly different question:

- **Histogram:** counts per bin. Bin width changes the story: too few bins hide the shape; too many create noise.
- **Kernel density estimate (KDE):** a smooth estimate of the density. Its bandwidth is a choice, much like bin width.
- **ECDF (empirical cumulative distribution function):** the proportion of values at or below each x. It needs no binning, and percentiles are easy to read from it.
- **Box plot:** median, quartiles and whiskers at 1.5 × IQR, with points beyond them shown individually. Good for comparing many groups.
- **Violin plot:** a mirrored KDE. It shows the shape (for example, two peaks) that a box plot hides.

First, the effect of bin width on `units_sold`:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharex=True, sharey=True)
for ax, bins in zip(axes, [5, 'auto', 60]):
    ax.hist(df['units_sold'], bins=bins, color=OKABE_ITO[0], edgecolor='white')
    ax.set_title(f'bins = {bins}')
    ax.set_xlabel('Units sold per record')
axes[0].set_ylabel('Number of records')
fig.suptitle('Histogram of units sold: the same data, three bin choices', y=1.03)
fig.tight_layout()

**Observation:** With 5 bins the shape is coarse; with 60 bins individual records dominate. The `auto` rule sits between the two. The distribution is right-skewed, with most records selling modest quantities and a few bulk orders forming a long tail.

Next, a KDE and an ECDF of the same variable, with the median marked:

In [ ]:
units = df['units_sold']
median_units = units.median()

fig, (ax_kde, ax_ecdf) = plt.subplots(1, 2, figsize=(10, 4))

sns.kdeplot(x=units, ax=ax_kde, fill=True, color=OKABE_ITO[0])
ax_kde.axvline(median_units, color='black', linestyle='--', linewidth=1, label=f'Median = {median_units:.0f}')
ax_kde.set_title('Kernel density estimate')
ax_kde.set_xlabel('Units sold per record')
ax_kde.legend(frameon=False)

ordered = np.sort(units.to_numpy())
proportion = np.arange(1, len(ordered) + 1) / len(ordered)
ax_ecdf.step(ordered, proportion, where='post', color=OKABE_ITO[3])
ax_ecdf.axhline(0.5, color='grey', linestyle=':', linewidth=1)
ax_ecdf.set_title('Empirical cumulative distribution')
ax_ecdf.set_xlabel('Units sold per record')
ax_ecdf.set_ylabel('Proportion of records at or below x')

Finally, a box plot and a violin plot compare `price` across categories. Categories are ordered by median price, and colours are consistent across the notebook.

In [ ]:
order = df.groupby('category')['price'].median().sort_values().index

fig, (ax_box, ax_violin) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

sns.boxplot(data=df, x='category', y='price', order=order, hue='category',
            palette=CATEGORY_COLOURS, legend=False, ax=ax_box)
ax_box.set_title('Box plot: median, quartiles and outliers')
ax_box.set_xlabel('Category')
ax_box.set_ylabel('Price (£)')

sns.violinplot(data=df, x='category', y='price', order=order, hue='category',
               palette=CATEGORY_COLOURS, legend=False, inner='quartile', ax=ax_violin)
ax_violin.set_title('Violin plot: shape of each distribution')
ax_violin.set_xlabel('Category')
ax_violin.set_ylabel('')

**Observation:** The KDE and ECDF agree that most records sell a moderate number of units, with the median below the mean because of the bulk orders. The price box plots differ clearly by category in both level and spread, and the violins show where most prices sit. Points beyond the whiskers are real records, not errors. Remove them only after you have checked them.

**Question:** Would removing the bulk orders change the story about typical sales? What would you report alongside the mean?

## 6. Relationships and trends

- **Scatter plots** show every record. With many points they overplot, so use transparency (`alpha`) and small markers.
- **Hexbin plots** (or 2D histograms) count points in cells. They suit dense data.
- **Line charts** are for *ordered* x-values such as dates. Never connect unordered categories with a line.
- **Association is not causation.** A negative relationship between price and units sold describes the data; it does not show that a price rise would reduce sales.

A scatter plot of every record, coloured by category:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for category, group in df.groupby('category'):
    ax.scatter(group['price'], group['units_sold'], s=30, alpha=0.6, edgecolor='white', linewidth=0.4,
               color=CATEGORY_COLOURS[category], label=category)
ax.set_title('Price versus units sold, by category')
ax.set_xlabel('Price (£)')
ax.set_ylabel('Units sold per record')
ax.legend(title='Category', frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1))

Dense data needs a different treatment. The next cell uses a larger synthetic sample (50,000 points) that follows the same price–units relationship. The x-axis is limited to £250 for legibility. Any report that truncates a range should say so.

In [ ]:
dense_rng = np.random.default_rng(42)
n_dense = 50_000
dense_price = dense_rng.lognormal(mean=np.log(45), sigma=0.9, size=n_dense)
dense_units = dense_rng.poisson(60 * np.exp(-dense_price / 90) + 8)

fig, (ax_scatter, ax_hex) = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)

ax_scatter.scatter(dense_price, dense_units, s=2, alpha=0.05, color=OKABE_ITO[0])
ax_scatter.set_title('Scatter: points overplot')
ax_scatter.set_ylabel('Units sold per record')

hexes = ax_hex.hexbin(dense_price, dense_units, gridsize=35, cmap='Blues', mincnt=1, bins='log')
fig.colorbar(hexes, ax=ax_hex, label='Number of records (log scale)')
ax_hex.set_title('Hexbin: density is visible')

for ax in (ax_scatter, ax_hex):
    ax.set_xlim(0, 250)
    ax.set_xlabel('Price (£)')
fig.tight_layout()

Last, an ordered axis: weekly revenue through 2025, with a 4-week rolling mean to reduce noise. Dates have a natural order, so a line is appropriate here.

In [ ]:
weekly_revenue = df.set_index('sale_date')['revenue'].resample('W').sum()
rolling_mean = weekly_revenue.rolling(window=4, center=True).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(weekly_revenue.index, weekly_revenue / 1000, color=OKABE_ITO[0], alpha=0.4, linewidth=1,
        label='Weekly total')
ax.plot(rolling_mean.index, rolling_mean / 1000, color=OKABE_ITO[3], linewidth=2.5,
        label='4-week rolling mean')
ax.set_title('Weekly revenue across 2025, in date order')
ax.set_xlabel('Week ending')
ax.set_ylabel('Revenue (£ thousands)')
ax.legend(frameon=False)

**Observation:** Units sold fall as price rises, and the cheaper categories overlap heavily. The hexbin reveals the dense band that the scatter plot hides. The weekly series has spikes, and the rolling mean shows the higher level in the later part of the year. The spikes may come from bulk orders; this chart alone does not establish that.

**Question:** Is the price–units pattern linear? Try plotting `np.log(price)` on the x-axis. Does the pattern look different, and what does that tell you about choosing a scale?

## 7. Visual integrity

A chart can use correct numbers and still mislead. Check for:

- **Truncated axes** on bar charts. Bar length is the encoding, so the baseline must be zero.
- **Scale choice.** A logarithmic axis suits data spanning orders of magnitude (for example, prices here range from under £3 to over £500). Label it clearly, because readers instinctively read distances as linear.
- **Colour type.** Use *sequential* palettes (light to dark) for ordered values, *diverging* palettes for values around a meaningful midpoint (for example, profit and loss), and *qualitative* palettes for unordered categories. Rainbow scales are hard to order and hard to read for many people.
- **Accessibility.** Roughly 1 in 12 men has some form of colour-vision deficiency. The Okabe–Ito palette used here helps, but do not rely on colour alone to carry meaning.
- **Labels and annotations.** State units, the sample size and anything excluded.

The first comparison uses regional revenue. The highest region's total is about 30% above the lowest, but a truncated axis makes that gap look far larger.

In [ ]:
region_revenue = df.groupby('region')['revenue'].sum().sort_index() / 1000

fig, (ax_bad, ax_good) = plt.subplots(1, 2, figsize=(11, 4))

ax_bad.bar(region_revenue.index, region_revenue.values, color=OKABE_ITO[0])
ax_bad.set_ylim(region_revenue.min() * 0.95, region_revenue.max() * 1.01)   # truncated baseline
ax_bad.set_title('Misleading: truncated y-axis')
ax_bad.set_ylabel('Revenue (£ thousands)')

bars = ax_good.bar(region_revenue.index, region_revenue.values, color=OKABE_ITO[0])
ax_good.bar_label(bars, labels=[f'£{v:,.0f}k' for v in region_revenue.values], padding=3)
ax_good.set_ylim(0, region_revenue.max() * 1.15)                             # zero baseline
ax_good.set_title('Improved: zero baseline and labelled values')
ax_good.set_ylabel('Revenue (£ thousands)')

The second comparison uses mean revenue for each category and region. The rainbow scale is hard to order, and the unlabelled cells give no exact values.

In [ ]:
pivot_revenue = df.pivot_table(index='category', columns='region', values='revenue', aggfunc='mean').round(0)

fig, (ax_bad, ax_good) = plt.subplots(1, 2, figsize=(11, 4))

sns.heatmap(pivot_revenue, cmap='jet', ax=ax_bad, cbar_kws={'label': 'Mean revenue (£)'})
ax_bad.set_title('Poor: rainbow scale for ordered values, no values shown')

sns.heatmap(pivot_revenue, cmap='Blues', annot=True, fmt='.0f', ax=ax_good,
            cbar_kws={'label': 'Mean revenue (£)'})
ax_good.set_title('Improved: sequential scale with labelled values')

for ax in (ax_bad, ax_good):
    ax.set_xlabel('Region')
    ax.set_ylabel('Category')

**Observation:** The truncated axis makes a gap of about 30% look like a several-fold difference, while the zero-baseline version shows its actual size. In the rainbow heatmap it is difficult to tell which cells are higher, whereas the sequential scale with values reads directly.

**Question:** Think of a chart you have seen in a report or news article that used a truncated axis or a rainbow scale. What would be the simplest fix?

## 8. Guided practical exercise

Work through three retail questions. For each one, choose a chart type that fits the variable types and the question, then complete the starter code. The starter cells run as they are, but they contain `TODO` placeholders for you to replace.

For each chart, check that it:

- answers the question stated in the cell;
- uses units and an informative title;
- avoids misleading scales and colours.

**Expected observations (not solutions):**

- *Question 1:* a zero-baseline comparison of four regions should make the size of the gaps easy to read. Check whether a difference in totals reflects a difference in the number of records per region before calling it meaningful.
- *Question 2:* a distribution chart for several groups should reveal differences in typical price, spread and outliers. Ordering the categories makes comparison easier.
- *Question 3:* a relationship chart should show the direction and strength of the pattern, and make clear where points overlap. Describe the pattern; do not claim causation.

In [ ]:
# Question 1: Which regions contribute most to revenue, and how large are the gaps?
# TODO 1: decide whether totals or means answer the question, and explain your choice in a comment.
region_revenue = df.groupby('region')['revenue'].sum()   # TODO: change the aggregation if needed

fig, ax = plt.subplots(figsize=(7, 4))
# TODO 2: draw the chart on a zero baseline and label the values
ax.set_title('TODO: a title that states the message')
ax.set_xlabel('TODO: region')
ax.set_ylabel('TODO: revenue, with units')

In [ ]:
# Question 2: How do price distributions differ between categories?
# TODO 1: choose a distribution chart that works for several groups (box, violin or ECDF lines).
# TODO 2: order the categories sensibly and use CATEGORY_COLOURS for consistency.
fig, ax = plt.subplots(figsize=(8, 4))
# TODO 3: draw the chart using df, with price on the value axis
ax.set_title('TODO: a title that states the message')
ax.set_xlabel('TODO: category')
ax.set_ylabel('TODO: price, with units')

In [ ]:
# Question 3: Is there a relationship between price and units sold, and how dense is the data?
# TODO 1: choose a relationship chart; decide whether transparency or a hexbin suits the density.
# TODO 2: label both axes with units.
fig, ax = plt.subplots(figsize=(8, 4))
# TODO 3: draw the chart using df
ax.set_title('TODO: a title that states the message')

## 9. Reflection and key takeaways

Answer these in your own words, in a markdown cell or with your partner:

1. Which analytical question does a histogram answer better than a bar chart, and which does a bar chart answer better?
2. What did Anscombe's quartet teach you about relying on summary statistics?
3. Name two ways a chart can mislead even when every number is correct.
4. When would you use a logarithmic scale, and what must you add so that readers understand it?
5. Which of your observations would need further analysis or more data before you could make a causal claim?

**Key takeaways**

- Start from the question and the variable types; choose the chart second.
- Use position and length for comparisons; avoid pie charts for fine comparisons.
- Plot distributions before summarising them, and check outliers before removing them.
- Use zero baselines for bars, sequential colours for ordered values, and labels with units.
- Exploratory charts can be rough; explanatory charts must be checked for misleading features.